# 08_05 EES Framework – Regulatory Alignment Score

## Purpose

This notebook calculates the Regulatory Alignment pillar of the Explainability-Ethics Scoring (EES) framework.

It is configurable across German Credit, Pima Diabetes, and COMPAS. Only `ACTIVE_DATASET` should be changed.

## Revised Regulatory Alignment Definition

Regulatory Alignment is treated as a structured research governance evidence score, not a legal compliance certification.

To avoid numerical double counting with the other EES pillars, the revised score uses four independent binary evidence criteria:

1. Transparency
2. Explanation Availability
3. Sensitive Feature Awareness
4. Documentation and Auditability

Fidelity, Stability, and Counterfactual Actionability are evaluated independently elsewhere in the EES framework and are therefore not re-used in this pillar.

Each retained criterion is scored 1 when the predefined evidence is present and 0 otherwise. The final Regulatory Alignment score is the proportion of the four criteria satisfied.


In [1]:
from google.colab import drive

drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
# =========================================================
# 0. Dataset Configuration
# =========================================================

# ACTIVE_DATASET = "german_credit"
# ACTIVE_DATASET = "pima_diabetes"
ACTIVE_DATASET = "compas"

DATASET_CONFIGS = {
    "german_credit": {
        "dataset_name": "german_credit",
        "dataset_label": "German Credit",
        "results_folder": "german_credit",
        "ees_input_folder": "08_01_ees_framework_prepare_inputs_german_credit",
        "fidelity_folder": "08_02_ees_framework_fidelity_score_german_credit",
        "stability_folder": "08_03_ees_framework_stability_score_german_credit",
        "actionability_folder": "08_04_ees_framework_actionability_score_german_credit",
        "regulatory_folder": "08_05_ees_framework_regulatory_alignment_score_german_credit",
        "sensitive_features": ["Age", "Sex"],
        "domain": "credit risk"
    },
    "pima_diabetes": {
        "dataset_name": "pima_diabetes",
        "dataset_label": "Pima Diabetes",
        "results_folder": "pima_diabetes",
        "ees_input_folder": "08_01_ees_framework_prepare_inputs_pima_diabetes",
        "fidelity_folder": "08_02_ees_framework_fidelity_score_pima_diabetes",
        "stability_folder": "08_03_ees_framework_stability_score_pima_diabetes",
        "actionability_folder": "08_04_ees_framework_actionability_score_pima_diabetes",
        "regulatory_folder": "08_05_ees_framework_regulatory_alignment_score_pima_diabetes",
        "sensitive_features": ["Age"],
        "domain": "healthcare risk"
    },
    "compas": {
        "dataset_name": "compas",
        "dataset_label": "COMPAS",
        "results_folder": "compas",
        "ees_input_folder": "08_01_ees_framework_prepare_inputs_compas",
        "fidelity_folder": "08_02_ees_framework_fidelity_score_compas",
        "stability_folder": "08_03_ees_framework_stability_score_compas",
        "actionability_folder": "08_04_ees_framework_actionability_score_compas",
        "regulatory_folder": "08_05_ees_framework_regulatory_alignment_score_compas",
        "sensitive_features": ["age", "sex", "race"],
        "domain": "criminal justice risk"
    }
}

CONFIG = DATASET_CONFIGS[ACTIVE_DATASET]
print("Active dataset:", CONFIG["dataset_label"])
print("Domain:", CONFIG["domain"])

Active dataset: COMPAS
Domain: criminal justice risk


In [3]:
# =========================================================
# 1. Import Required Libraries
# =========================================================

import warnings
warnings.filterwarnings("ignore")

from pathlib import Path
from datetime import datetime
import json
import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 250)
pd.set_option("display.float_format", lambda x: f"{x:.6f}")

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
print("Libraries imported successfully.")

Libraries imported successfully.


In [4]:
# =========================================================
# 2. Mount Google Drive and Configure Paths
# =========================================================


ROOT_PATH = Path("/content/drive/MyDrive/Master_Research")

# Stop immediately if the real Drive is not available
if not ROOT_PATH.exists():
    raise RuntimeError(
        "Google Drive is not mounted correctly or Master_Research "
        "is not accessible. Do not continue."
    )

DATASET_NAME = CONFIG["dataset_name"]
NOTEBOOK_NAME = CONFIG["regulatory_folder"]

RESULTS_ROOT = ROOT_PATH / "07_Results" / CONFIG["results_folder"]

EES_INPUT_RESULTS_PATH = RESULTS_ROOT / CONFIG["ees_input_folder"]
EES_INPUT_TABLES_PATH = EES_INPUT_RESULTS_PATH / "tables"
EES_INPUT_DATASETS_PATH = EES_INPUT_RESULTS_PATH / "datasets"

FIDELITY_RESULTS_PATH = RESULTS_ROOT / CONFIG["fidelity_folder"]
FIDELITY_TABLES_PATH = FIDELITY_RESULTS_PATH / "tables"
FIDELITY_DATASETS_PATH = FIDELITY_RESULTS_PATH / "datasets"

STABILITY_RESULTS_PATH = RESULTS_ROOT / CONFIG["stability_folder"]
STABILITY_TABLES_PATH = STABILITY_RESULTS_PATH / "tables"
STABILITY_DATASETS_PATH = STABILITY_RESULTS_PATH / "datasets"

ACTIONABILITY_RESULTS_PATH = RESULTS_ROOT / CONFIG["actionability_folder"]
ACTIONABILITY_TABLES_PATH = ACTIONABILITY_RESULTS_PATH / "tables"
ACTIONABILITY_DATASETS_PATH = ACTIONABILITY_RESULTS_PATH / "datasets"

RESULTS_PATH = RESULTS_ROOT / NOTEBOOK_NAME
TABLES_PATH = RESULTS_PATH / "tables"
DATASETS_PATH = RESULTS_PATH / "datasets"
LOGS_PATH = RESULTS_PATH / "logs"

for path in [RESULTS_PATH, TABLES_PATH, DATASETS_PATH, LOGS_PATH]:
    path.mkdir(parents=True, exist_ok=True)

print("Google Drive connected successfully.")
print("Master Research path:", ROOT_PATH)
print("Regulatory output path:", RESULTS_PATH)

Google Drive connected successfully.
Master Research path: /content/drive/MyDrive/Master_Research
Regulatory output path: /content/drive/MyDrive/Master_Research/07_Results/compas/08_05_ees_framework_regulatory_alignment_score_compas


In [5]:
# =========================================================
# 3. Helper Functions
# =========================================================

def read_csv_safely(path, label, required=True):
    path = Path(path)
    if not path.exists():
        if required:
            raise FileNotFoundError(f"{label} not found at: {path}")
        print(f"Optional file not found: {label} at {path}")
        return pd.DataFrame()
    return pd.read_csv(path)


def find_first_existing_file(candidate_paths, label, required=True):
    for path in candidate_paths:
        path = Path(path)
        if path.exists():
            print(f"{label} found at: {path}")
            return path

    if required:
        checked = "\\n".join(str(path) for path in candidate_paths)
        raise FileNotFoundError(f"{label} not found. Checked:\\n{checked}")

    print(f"{label} not found.")
    return None


def save_df(df, filename, folder=None, index=False):
    if folder is None:
        folder = TABLES_PATH
    folder = Path(folder)
    folder.mkdir(parents=True, exist_ok=True)
    output_path = folder / filename
    df.to_csv(output_path, index=index)
    print(f"Saved: {output_path}")
    return output_path


def save_excel(dfs, filename, folder=None):
    if folder is None:
        folder = TABLES_PATH
    folder = Path(folder)
    folder.mkdir(parents=True, exist_ok=True)
    output_path = folder / filename

    with pd.ExcelWriter(output_path, engine="openpyxl") as writer:
        for sheet_name, df in dfs.items():
            if isinstance(df, pd.DataFrame):
                df.to_excel(writer, sheet_name=sheet_name[:31], index=False)

    print(f"Saved: {output_path}")
    return output_path


def evidence_status(score):
    if pd.isna(score):
        return "Not available"
    return f"{int(round(score * 4))} of 4 predefined governance evidence criteria satisfied"


## 4. Load EES Component Inputs

This notebook loads the master EES input table and component scores from Fidelity, Stability, and Actionability.

In [6]:
from pathlib import Path

print("EES_INPUT_DATASETS_PATH:")
print(EES_INPUT_DATASETS_PATH)

print("\nDoes folder exist?")
print(EES_INPUT_DATASETS_PATH.exists())

print("\nExpected master file:")
expected = EES_INPUT_DATASETS_PATH / "master_ees_input.csv"
print(expected)
print("File exists:", expected.exists())

print("\nFiles visible to Python in that folder:")
if EES_INPUT_DATASETS_PATH.exists():
    for f in EES_INPUT_DATASETS_PATH.iterdir():
        print(repr(f.name))
else:
    print("FOLDER NOT VISIBLE TO CURRENT RUNTIME")

EES_INPUT_DATASETS_PATH:
/content/drive/MyDrive/Master_Research/07_Results/compas/08_01_ees_framework_prepare_inputs_compas/datasets

Does folder exist?
True

Expected master file:
/content/drive/MyDrive/Master_Research/07_Results/compas/08_01_ees_framework_prepare_inputs_compas/datasets/master_ees_input.csv
File exists: True

Files visible to Python in that folder:
'model_inventory.csv'
'input_availability.csv'
'model_input_summary.csv'
'master_ees_input_table.csv'
'baseline_performance.csv'
'shap_all_test_long.csv'
'lime_all_test_long.csv'
'shap_lime_disagreement.csv'
'dice_counterfactuals.csv'
'actionability_summary.csv'
'ees_model_performance_input.csv'
'ees_shap_long_input.csv'
'ees_shap_reconstruction_input.csv'
'ees_lime_long_input.csv'
'ees_lime_fidelity_input.csv'
'ees_shap_lime_agreement_input.csv'
'ees_actionability_input.csv'
'master_ees_input.csv'


In [7]:
# =========================================================
# 4. Load EES Component Inputs
# =========================================================

master_input_file = find_first_existing_file(
    [
        EES_INPUT_DATASETS_PATH / "master_ees_input.csv",
        EES_INPUT_TABLES_PATH / "master_ees_input.csv"
    ],
    "Master EES input"
)

fidelity_file = find_first_existing_file(
    [
        FIDELITY_DATASETS_PATH / "ees_fidelity_model_scores.csv",
        FIDELITY_TABLES_PATH / "ees_fidelity_model_scores.csv",
        FIDELITY_DATASETS_PATH / "fidelity_score_summary.csv",
        FIDELITY_TABLES_PATH / "fidelity_score_summary.csv"
    ],
    "Fidelity model scores",
    required=False
)

stability_file = find_first_existing_file(
    [
        STABILITY_DATASETS_PATH / "ees_stability_model_scores.csv",
        STABILITY_TABLES_PATH / "ees_stability_model_scores.csv",
        STABILITY_DATASETS_PATH / "stability_score_summary.csv",
        STABILITY_TABLES_PATH / "stability_score_summary.csv"
    ],
    "Stability model scores",
    required=False
)

actionability_file = find_first_existing_file(
    [
        ACTIONABILITY_DATASETS_PATH / "ees_actionability_model_scores.csv",
        ACTIONABILITY_TABLES_PATH / "ees_actionability_model_scores.csv",
        ACTIONABILITY_DATASETS_PATH / "actionability_score_summary.csv",
        ACTIONABILITY_TABLES_PATH / "actionability_score_summary.csv"
    ],
    "Actionability model scores",
    required=False
)

master_ees_input_df = read_csv_safely(master_input_file, "Master EES input")
fidelity_model_scores_df = read_csv_safely(fidelity_file, "Fidelity model scores", required=False) if fidelity_file else pd.DataFrame()
stability_model_scores_df = read_csv_safely(stability_file, "Stability model scores", required=False) if stability_file else pd.DataFrame()
actionability_model_scores_df = read_csv_safely(actionability_file, "Actionability model scores", required=False) if actionability_file else pd.DataFrame()

print("master_ees_input_df:", master_ees_input_df.shape)
print("fidelity_model_scores_df:", fidelity_model_scores_df.shape)
print("stability_model_scores_df:", stability_model_scores_df.shape)
print("actionability_model_scores_df:", actionability_model_scores_df.shape)

display(master_ees_input_df.head())

Master EES input found at: /content/drive/MyDrive/Master_Research/07_Results/compas/08_01_ees_framework_prepare_inputs_compas/datasets/master_ees_input.csv
Fidelity model scores found at: /content/drive/MyDrive/Master_Research/07_Results/compas/08_02_ees_framework_fidelity_score_compas/datasets/ees_fidelity_model_scores.csv
Stability model scores found at: /content/drive/MyDrive/Master_Research/07_Results/compas/08_03_ees_framework_stability_score_compas/datasets/ees_stability_model_scores.csv
Actionability model scores found at: /content/drive/MyDrive/Master_Research/07_Results/compas/08_04_ees_framework_actionability_score_compas/datasets/ees_actionability_model_scores.csv
master_ees_input_df: (5, 25)
fidelity_model_scores_df: (5, 18)
stability_model_scores_df: (5, 15)
actionability_model_scores_df: (5, 20)


,model_id,model_name,dataset_name,dataset_label,positive_class_label,negative_class_label,accuracy,precision,recall,f1_score,roc_auc,average_precision,business_priority_score,business_priority_rank,shap_lime_agreement_score,shap_lime_agreement_score_pct,mean_kendall_tau,mean_top_k_overlap_ratio,availability_score,proximity_score,sparsity_score,actionability_score,actionability_rank,mean_min_normalized_numeric_distance,mean_changed_features
0,decision_tree,Decision Tree,compas,COMPAS,Recidivism,No recidivism,0.674857,0.655822,0.682304,0.668801,0.723393,0.673207,0.690795,4,0.511505,51.150489,0.023010,0.448543,1.000000,0.959375,0.892237,0.950537,1,0.040625,1.077626
1,logistic_regression,Logistic Regression,compas,COMPAS,Recidivism,No recidivism,0.673714,0.670226,0.633610,0.651404,0.726255,0.686058,0.681653,5,0.336659,33.665877,-0.326682,0.341257,1.000000,0.958794,0.887626,0.948806,2,0.041206,1.123744
2,mlp,MLP,compas,COMPAS,Recidivism,No recidivism,0.676000,0.661955,0.667458,0.664695,0.739305,0.693323,0.694546,3,0.345068,34.506810,-0.309864,0.288286,1.000000,0.950526,0.881861,0.944129,4,0.049474,1.181390
3,random_forest,Random Forest,compas,COMPAS,Recidivism,No recidivism,0.824857,0.807582,0.834917,0.821022,0.903823,0.901771,0.854279,1,0.439104,43.910428,-0.121791,0.383429,1.000000,0.956268,0.875014,0.943761,5,0.043732,1.249856
4,xgboost,XGBoost,compas,COMPAS,Recidivism,No recidivism,0.691143,0.680431,0.675178,0.677794,0.760197,0.735896,0.710761,2,0.435232,43.523177,-0.129536,0.428143,1.000000,0.955618,0.882705,0.946108,3,0.044382,1.172950


## 5. Build Regulatory Evidence Table

In [8]:
# =========================================================
# 5. Build Regulatory Evidence Table
# =========================================================

regulatory_evidence_df = master_ees_input_df[["model_id", "model_name"]].drop_duplicates().copy()

if not fidelity_model_scores_df.empty:
    fidelity_cols = [col for col in [
        "model_id", "fidelity_score", "shap_reconstruction_fidelity_score",
        "lime_fidelity_score", "shap_lime_agreement_score"
    ] if col in fidelity_model_scores_df.columns]
    if len(fidelity_cols) > 1:
        regulatory_evidence_df = regulatory_evidence_df.merge(
            fidelity_model_scores_df[fidelity_cols], on="model_id", how="left"
        )

if not stability_model_scores_df.empty:
    stability_cols = [col for col in [
        "model_id", "stability_score", "mean_local_lipschitz_ratio"
    ] if col in stability_model_scores_df.columns]
    if len(stability_cols) > 1:
        regulatory_evidence_df = regulatory_evidence_df.merge(
            stability_model_scores_df[stability_cols], on="model_id", how="left"
        )

if not actionability_model_scores_df.empty:
    actionability_cols = [col for col in [
        "model_id", "actionability_score", "availability_score",
        "proximity_score", "sparsity_score"
    ] if col in actionability_model_scores_df.columns]
    if len(actionability_cols) > 1:
        regulatory_evidence_df = regulatory_evidence_df.merge(
            actionability_model_scores_df[actionability_cols], on="model_id", how="left"
        )

regulatory_evidence_df["has_fidelity_evidence"] = (
    regulatory_evidence_df["fidelity_score"].notna()
    if "fidelity_score" in regulatory_evidence_df.columns
    else False
)
regulatory_evidence_df["has_stability_evidence"] = (
    regulatory_evidence_df["stability_score"].notna()
    if "stability_score" in regulatory_evidence_df.columns
    else False
)
regulatory_evidence_df["has_actionability_evidence"] = (
    regulatory_evidence_df["actionability_score"].notna()
    if "actionability_score" in regulatory_evidence_df.columns
    else False
)
regulatory_evidence_df["has_sensitive_feature_register"] = len(CONFIG["sensitive_features"]) > 0
regulatory_evidence_df["has_documented_pipeline"] = True
regulatory_evidence_df["has_saved_audit_outputs"] = True

display(regulatory_evidence_df)
save_df(regulatory_evidence_df, "regulatory_evidence_table.csv")

,model_id,model_name,fidelity_score,shap_reconstruction_fidelity_score,lime_fidelity_score,shap_lime_agreement_score,stability_score,mean_local_lipschitz_ratio,actionability_score,availability_score,proximity_score,sparsity_score,has_fidelity_evidence,has_stability_evidence,has_actionability_evidence,has_sensitive_feature_register,has_documented_pipeline,has_saved_audit_outputs
0,decision_tree,Decision Tree,0.591769,1.000000,0.263802,0.511505,1.000000,0.031236,0.950537,1.000000,0.959375,0.892237,True,True,True,True,True,True
1,logistic_regression,Logistic Regression,0.567808,1.000000,0.366766,0.336659,0.661564,0.132007,0.948806,1.000000,0.958794,0.887626,True,True,True,True,True,True
2,mlp,MLP,0.600626,1.000000,0.456811,0.345068,0.632455,0.140674,0.944129,1.000000,0.950526,0.881861,True,True,True,True,True,True
3,random_forest,Random Forest,0.614249,0.999444,0.404200,0.439104,0.000000,0.328991,0.943761,1.000000,0.956268,0.875014,True,True,True,True,True,True
4,xgboost,XGBoost,0.651325,1.000000,0.518743,0.435232,0.619616,0.144497,0.946108,1.000000,0.955618,0.882705,True,True,True,True,True,True


Saved: /content/drive/MyDrive/Master_Research/07_Results/compas/08_05_ees_framework_regulatory_alignment_score_compas/tables/regulatory_evidence_table.csv


PosixPath('/content/drive/MyDrive/Master_Research/07_Results/compas/08_05_ees_framework_regulatory_alignment_score_compas/tables/regulatory_evidence_table.csv')

## 6. Define Regulatory Alignment Rubric

The rubric uses seven equal-weight criteria. Equal weights are used to avoid introducing unsupported subjective preference among governance dimensions.

In [9]:
# =========================================================
# 6. Define Regulatory Alignment Rubric
# =========================================================

regulatory_rubric_df = pd.DataFrame([
    {
        "rubric_id": "R1",
        "rubric_name": "Transparency",
        "question": "Is there a documented modelling and explanation pipeline?",
        "score_logic": "1 if documented pipeline exists, else 0"
    },
    {
        "rubric_id": "R2",
        "rubric_name": "Explanation availability",
        "question": "Are explanation outputs available for model assessment?",
        "score_logic": "1 if Fidelity evidence exists, else 0"
    },
    {
        "rubric_id": "R3",
        "rubric_name": "Fidelity and explanation faithfulness",
        "question": "Do explanations faithfully reflect model behaviour?",
        "score_logic": "Use model-level Fidelity score"
    },
    {
        "rubric_id": "R4",
        "rubric_name": "Stability and robustness",
        "question": "Are model outputs stable under perturbation?",
        "score_logic": "Use model-level Stability score"
    },
    {
        "rubric_id": "R5",
        "rubric_name": "Actionability and contestability",
        "question": "Can unfavourable outcomes be contested through recourse evidence?",
        "score_logic": "Use model-level Actionability score"
    },
    {
        "rubric_id": "R6",
        "rubric_name": "Sensitive-feature awareness",
        "question": "Are sensitive features explicitly recognised?",
        "score_logic": "1 if sensitive feature register exists, else 0"
    },
    {
        "rubric_id": "R7",
        "rubric_name": "Documentation and auditability",
        "question": "Are outputs and assumptions saved for auditability?",
        "score_logic": "1 if standard outputs/manifests exist, else 0"
    }
])

display(regulatory_rubric_df)
save_df(regulatory_rubric_df, "regulatory_alignment_rubric.csv")

,rubric_id,rubric_name,question,score_logic
0,R1,Transparency,Is there a documented modelling and explanatio...,"1 if documented pipeline exists, else 0"
1,R2,Explanation availability,Are explanation outputs available for model as...,"1 if Fidelity evidence exists, else 0"
2,R3,Fidelity and explanation faithfulness,Do explanations faithfully reflect model behav...,Use model-level Fidelity score
3,R4,Stability and robustness,Are model outputs stable under perturbation?,Use model-level Stability score
4,R5,Actionability and contestability,Can unfavourable outcomes be contested through...,Use model-level Actionability score
5,R6,Sensitive-feature awareness,Are sensitive features explicitly recognised?,"1 if sensitive feature register exists, else 0"
6,R7,Documentation and auditability,Are outputs and assumptions saved for auditabi...,"1 if standard outputs/manifests exist, else 0"


Saved: /content/drive/MyDrive/Master_Research/07_Results/compas/08_05_ees_framework_regulatory_alignment_score_compas/tables/regulatory_alignment_rubric.csv


PosixPath('/content/drive/MyDrive/Master_Research/07_Results/compas/08_05_ees_framework_regulatory_alignment_score_compas/tables/regulatory_alignment_rubric.csv')

## 7. Calculate Model-Level Regulatory Alignment Score

In [10]:
# =========================================================
# 6. Calculate Revised Regulatory Alignment Score
# =========================================================

# Fidelity, Stability, and Actionability are excluded here because they are
# already independent EES pillars. This prevents numerical double counting.

regulatory_rows = []

for _, row in regulatory_evidence_df.iterrows():
    r1_transparency = 1.0 if bool(row.get("has_documented_pipeline", False)) else 0.0

    # Availability check only; the numerical Fidelity score is not reused.
    r2_explanation_availability = (
        1.0 if bool(row.get("has_fidelity_evidence", False)) else 0.0
    )

    r3_sensitive_feature_awareness = (
        1.0 if bool(row.get("has_sensitive_feature_register", False)) else 0.0
    )

    r4_documentation_auditability = (
        1.0 if bool(row.get("has_saved_audit_outputs", False)) else 0.0
    )

    criterion_values = [
        r1_transparency,
        r2_explanation_availability,
        r3_sensitive_feature_awareness,
        r4_documentation_auditability,
    ]

    regulatory_alignment_score = float(np.mean(criterion_values))
    criteria_satisfied = int(np.sum(criterion_values))

    regulatory_rows.append({
        "dataset_name": DATASET_NAME,
        "dataset_label": CONFIG["dataset_label"],
        "model_id": row["model_id"],
        "model_name": row["model_name"],
        "transparency_score": r1_transparency,
        "explanation_availability_score": r2_explanation_availability,
        "sensitive_feature_awareness_score": r3_sensitive_feature_awareness,
        "documentation_auditability_score": r4_documentation_auditability,
        "criteria_satisfied": criteria_satisfied,
        "criteria_total": 4,
        "regulatory_alignment_score": regulatory_alignment_score,
        "regulatory_alignment_score_pct": 100.0 * regulatory_alignment_score,
        "evidence_status": evidence_status(regulatory_alignment_score),
    })

regulatory_alignment_model_scores_df = pd.DataFrame(regulatory_rows)

# Ties remain ties; no artificial model ordering is imposed.
regulatory_alignment_model_scores_df["regulatory_alignment_rank"] = (
    regulatory_alignment_model_scores_df["regulatory_alignment_score"]
    .rank(method="min", ascending=False)
    .astype(int)
)

regulatory_alignment_model_scores_df = regulatory_alignment_model_scores_df.sort_values(
    ["regulatory_alignment_score", "model_name"],
    ascending=[False, True]
).reset_index(drop=True)

display(regulatory_alignment_model_scores_df)
save_df(regulatory_alignment_model_scores_df, "ees_regulatory_alignment_model_scores.csv", folder=DATASETS_PATH)
save_df(regulatory_alignment_model_scores_df, "ees_regulatory_alignment_model_scores.csv", folder=TABLES_PATH)


,dataset_name,dataset_label,model_id,model_name,transparency_score,explanation_availability_score,sensitive_feature_awareness_score,documentation_auditability_score,criteria_satisfied,criteria_total,regulatory_alignment_score,regulatory_alignment_score_pct,evidence_status,regulatory_alignment_rank
0,compas,COMPAS,decision_tree,Decision Tree,1.000000,1.000000,1.000000,1.000000,4,4,1.000000,100.000000,4 of 4 predefined governance evidence criteria...,1
1,compas,COMPAS,logistic_regression,Logistic Regression,1.000000,1.000000,1.000000,1.000000,4,4,1.000000,100.000000,4 of 4 predefined governance evidence criteria...,1
2,compas,COMPAS,mlp,MLP,1.000000,1.000000,1.000000,1.000000,4,4,1.000000,100.000000,4 of 4 predefined governance evidence criteria...,1
3,compas,COMPAS,random_forest,Random Forest,1.000000,1.000000,1.000000,1.000000,4,4,1.000000,100.000000,4 of 4 predefined governance evidence criteria...,1
4,compas,COMPAS,xgboost,XGBoost,1.000000,1.000000,1.000000,1.000000,4,4,1.000000,100.000000,4 of 4 predefined governance evidence criteria...,1


Saved: /content/drive/MyDrive/Master_Research/07_Results/compas/08_05_ees_framework_regulatory_alignment_score_compas/datasets/ees_regulatory_alignment_model_scores.csv
Saved: /content/drive/MyDrive/Master_Research/07_Results/compas/08_05_ees_framework_regulatory_alignment_score_compas/tables/ees_regulatory_alignment_model_scores.csv


PosixPath('/content/drive/MyDrive/Master_Research/07_Results/compas/08_05_ees_framework_regulatory_alignment_score_compas/tables/ees_regulatory_alignment_model_scores.csv')

## 8. Create Long-Format Rubric Evidence

In [11]:
# =========================================================
# 8. Create Long-Format Rubric Evidence
# =========================================================

# Revised Regulatory Alignment contains four independent
# governance-evidence criteria only.
rubric_mapping = [
    ("R1", "Transparency", "transparency_score"),
    ("R2", "Explanation availability", "explanation_availability_score"),
    ("R3", "Sensitive-feature awareness", "sensitive_feature_awareness_score"),
    ("R4", "Documentation and auditability", "documentation_auditability_score")
]

rubric_long_rows = []

for _, row in regulatory_alignment_model_scores_df.iterrows():
    for rubric_id, rubric_name, score_col in rubric_mapping:
        rubric_long_rows.append({
            "dataset_name": row["dataset_name"],
            "dataset_label": row["dataset_label"],
            "model_id": row["model_id"],
            "model_name": row["model_name"],
            "rubric_id": rubric_id,
            "rubric_name": rubric_name,
            "rubric_score": row[score_col]
        })

regulatory_alignment_long_df = pd.DataFrame(rubric_long_rows)

display(regulatory_alignment_long_df.head(20))

,dataset_name,dataset_label,model_id,model_name,rubric_id,rubric_name,rubric_score
0,compas,COMPAS,decision_tree,Decision Tree,R1,Transparency,1.000000
1,compas,COMPAS,decision_tree,Decision Tree,R2,Explanation availability,1.000000
2,compas,COMPAS,decision_tree,Decision Tree,R3,Sensitive-feature awareness,1.000000
3,compas,COMPAS,decision_tree,Decision Tree,R4,Documentation and auditability,1.000000
4,compas,COMPAS,logistic_regression,Logistic Regression,R1,Transparency,1.000000
5,compas,COMPAS,logistic_regression,Logistic Regression,R2,Explanation availability,1.000000
6,compas,COMPAS,logistic_regression,Logistic Regression,R3,Sensitive-feature awareness,1.000000
7,compas,COMPAS,logistic_regression,Logistic Regression,R4,Documentation and auditability,1.000000
8,compas,COMPAS,mlp,MLP,R1,Transparency,1.000000
9,compas,COMPAS,mlp,MLP,R2,Explanation availability,1.000000


## 9. Interpretation Table

In [12]:
# =========================================================
# 9. Evidence Interpretation Table
# =========================================================

regulatory_alignment_interpretation_df = pd.DataFrame([
    {"score": 0.00, "criteria_satisfied": "0 of 4", "interpretation": "No predefined governance evidence criteria satisfied."},
    {"score": 0.25, "criteria_satisfied": "1 of 4", "interpretation": "One predefined governance evidence criterion satisfied."},
    {"score": 0.50, "criteria_satisfied": "2 of 4", "interpretation": "Two predefined governance evidence criteria satisfied."},
    {"score": 0.75, "criteria_satisfied": "3 of 4", "interpretation": "Three predefined governance evidence criteria satisfied."},
    {"score": 1.00, "criteria_satisfied": "4 of 4", "interpretation": "All four predefined governance evidence criteria satisfied."},
])
display(regulatory_alignment_interpretation_df)
save_df(regulatory_alignment_interpretation_df, "regulatory_alignment_interpretation.csv")


,score,criteria_satisfied,interpretation
0,0.000000,0 of 4,No predefined governance evidence criteria sat...
1,0.250000,1 of 4,One predefined governance evidence criterion s...
2,0.500000,2 of 4,Two predefined governance evidence criteria sa...
3,0.750000,3 of 4,Three predefined governance evidence criteria ...
4,1.000000,4 of 4,All four predefined governance evidence criter...


Saved: /content/drive/MyDrive/Master_Research/07_Results/compas/08_05_ees_framework_regulatory_alignment_score_compas/tables/regulatory_alignment_interpretation.csv


PosixPath('/content/drive/MyDrive/Master_Research/07_Results/compas/08_05_ees_framework_regulatory_alignment_score_compas/tables/regulatory_alignment_interpretation.csv')

## 10. Save Regulatory Alignment Outputs

In [13]:
# =========================================================
# 10. Save Revised Regulatory Alignment Outputs
# =========================================================

# ---------------------------------------------------------
# Save CSV outputs to the main tables folder
# ---------------------------------------------------------
save_df(
    regulatory_alignment_model_scores_df,
    "ees_regulatory_alignment_model_scores.csv"
)

save_df(
    regulatory_alignment_model_scores_df,
    "regulatory_alignment_score_summary.csv"
)

save_df(
    regulatory_alignment_long_df,
    "regulatory_alignment_rubric_long.csv"
)

save_df(
    regulatory_evidence_df,
    "regulatory_evidence_table.csv"
)

# ---------------------------------------------------------
# Save downstream-compatible datasets
# ---------------------------------------------------------
save_df(
    regulatory_alignment_model_scores_df,
    "ees_regulatory_alignment_model_scores.csv",
    folder=DATASETS_PATH
)

save_df(
    regulatory_alignment_model_scores_df,
    "regulatory_alignment_score_summary.csv",
    folder=DATASETS_PATH
)

save_df(
    regulatory_alignment_long_df,
    "regulatory_alignment_rubric_long.csv",
    folder=DATASETS_PATH
)

save_df(
    regulatory_evidence_df,
    "regulatory_evidence_table.csv",
    folder=DATASETS_PATH
)

# ---------------------------------------------------------
# Save consolidated Excel workbook
# ---------------------------------------------------------
save_excel(
    {
        "regulatory_scores": regulatory_alignment_model_scores_df,
        "rubric_long": regulatory_alignment_long_df,
        "rubric": regulatory_rubric_df,
        "evidence_table": regulatory_evidence_df,
        "interpretation": regulatory_alignment_interpretation_df
    },
    f"ees_regulatory_alignment_score_{DATASET_NAME}_outputs.xlsx"
)

# ---------------------------------------------------------
# Save run manifest
# ---------------------------------------------------------
manifest = {
    "dataset_name": DATASET_NAME,
    "dataset_label": CONFIG["dataset_label"],
    "notebook_name": NOTEBOOK_NAME,
    "run_timestamp": datetime.now().isoformat(),
    "random_state": RANDOM_STATE,
    "domain": CONFIG["domain"],
    "sensitive_features": CONFIG["sensitive_features"],
    "regulatory_score_formula": (
        "Mean of four equal-weight binary evidence criteria: "
        "Transparency, Explanation Availability, "
        "Sensitive Feature Awareness, and Documentation and Auditability."
    ),
    "caution": (
        "This is a structured governance evidence indicator, not legal "
        "compliance certification. Fidelity, Stability, and Actionability "
        "are excluded to prevent numerical double counting."
    ),
    "standardisation_note": (
        "This regulatory alignment notebook uses the same configurable "
        "four-criterion rubric across datasets."
    )
}

with open(
    LOGS_PATH / "ees_regulatory_alignment_score_manifest.json",
    "w",
    encoding="utf-8"
) as f:
    json.dump(manifest, f, indent=4)

with open(
    TABLES_PATH / "ees_regulatory_alignment_score_manifest.json",
    "w",
    encoding="utf-8"
) as f:
    json.dump(manifest, f, indent=4)

print("All revised regulatory alignment outputs saved.")

Saved: /content/drive/MyDrive/Master_Research/07_Results/compas/08_05_ees_framework_regulatory_alignment_score_compas/tables/ees_regulatory_alignment_model_scores.csv
Saved: /content/drive/MyDrive/Master_Research/07_Results/compas/08_05_ees_framework_regulatory_alignment_score_compas/tables/regulatory_alignment_score_summary.csv
Saved: /content/drive/MyDrive/Master_Research/07_Results/compas/08_05_ees_framework_regulatory_alignment_score_compas/tables/regulatory_alignment_rubric_long.csv
Saved: /content/drive/MyDrive/Master_Research/07_Results/compas/08_05_ees_framework_regulatory_alignment_score_compas/tables/regulatory_evidence_table.csv
Saved: /content/drive/MyDrive/Master_Research/07_Results/compas/08_05_ees_framework_regulatory_alignment_score_compas/datasets/ees_regulatory_alignment_model_scores.csv
Saved: /content/drive/MyDrive/Master_Research/07_Results/compas/08_05_ees_framework_regulatory_alignment_score_compas/datasets/regulatory_alignment_score_summary.csv
Saved: /content/d

## 11. Interpretation Note

The revised Regulatory Alignment pillar measures the presence of four predefined governance evidence criteria: Transparency, Explanation Availability, Sensitive Feature Awareness, and Documentation and Auditability.

Fidelity, Stability, and Counterfactual Actionability are not reused in this score because they are evaluated independently as EES pillars. This removes numerical double counting.

The score is interpreted as the proportion of the four specified evidence criteria satisfied. It is a structured research governance indicator, not a legal compliance decision.


In [14]:
# =========================================================
# 11. Final Completion Message
# =========================================================

print("=" * 100)
print(f"08_05 Regulatory Alignment Score completed successfully for: {CONFIG['dataset_label']}")
print("=" * 100)

print(f"Tables saved to:   {TABLES_PATH}")
print(f"Datasets saved to: {DATASETS_PATH}")
print(f"Logs saved to:     {LOGS_PATH}")

print("\nKey outputs:")
print("- ees_regulatory_alignment_model_scores.csv")
print("- regulatory_alignment_score_summary.csv")
print("- regulatory_alignment_rubric_long.csv")
print("- regulatory_evidence_table.csv")
print(f"- ees_regulatory_alignment_score_{DATASET_NAME}_outputs.xlsx")

08_05 Regulatory Alignment Score completed successfully for: COMPAS
Tables saved to:   /content/drive/MyDrive/Master_Research/07_Results/compas/08_05_ees_framework_regulatory_alignment_score_compas/tables
Datasets saved to: /content/drive/MyDrive/Master_Research/07_Results/compas/08_05_ees_framework_regulatory_alignment_score_compas/datasets
Logs saved to:     /content/drive/MyDrive/Master_Research/07_Results/compas/08_05_ees_framework_regulatory_alignment_score_compas/logs

Key outputs:
- ees_regulatory_alignment_model_scores.csv
- regulatory_alignment_score_summary.csv
- regulatory_alignment_rubric_long.csv
- regulatory_evidence_table.csv
- ees_regulatory_alignment_score_compas_outputs.xlsx
